# RecipeNLG Data Exploration

This notebook consolidates the exploratory analysis from earlier notebooks into a single workflow.

It focuses on:
- loading and profiling the raw dataset,
- identifying duplicate and suspicious records,
- analyzing ingredient and direction lengths,
- normalizing titles, ingredients, and directions,
- comparing the dataset before and after cleaning.


In [ ]:
from pathlib import Path
import ast
import math
import re
from collections import Counter, OrderedDict
from fractions import Fraction
from typing import List, Optional, Union

import matplotlib.pyplot as plt
import pandas as pd
import regex
import unidecode

DATASET_CANDIDATES = [
    Path('../data/raw/RecipeNLG/RecipeNLG_dataset.csv'),
    Path('..') / 'data' / 'raw' / 'RecipeNLG' / 'RecipeNLG_dataset.csv',
]

dataset_path = next((p for p in DATASET_CANDIDATES if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError('RecipeNLG_dataset.csv was not found in the expected locations.')

df = pd.read_csv(dataset_path)
if 'Unnamed: 0' in df.columns:
    df = df.rename(columns={'Unnamed: 0': 'recipe_id'})

print('dataset_path:', dataset_path)
print('shape:', df.shape)
df.head()



## 1. Raw Dataset Overview

The first pass checks the dataset structure and highlights that recipe titles are not unique identifiers.


In [ ]:
df.info()
display(df.head(3))
display(df['title'].value_counts().head(20))
display(df[df['title'] == 'Chicken Casserole'].head())


## 2. Ingredient Frequency Analysis

This section collects all ingredients from the `NER` column and counts the most common items across the corpus.


In [ ]:
def safe_parse_list(value):
    # Accept already-parsed lists so downstream code can handle mixed input types.
    if isinstance(value, list):
        return value
    if not isinstance(value, str):
        return []
    try:
        # `ast.literal_eval` safely parses stringified Python lists from the dataset.
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (ValueError, SyntaxError):
        return []

all_ingredients = []
for ingredients_list in df['NER'].dropna():
    all_ingredients.extend(safe_parse_list(ingredients_list))

ingredient_counts = Counter(all_ingredients)
ingredient_df = pd.DataFrame(ingredient_counts.items(), columns=['Ingredient', 'Count'])
ingredient_df = ingredient_df.sort_values('Count', ascending=False).reset_index(drop=True)

print(f'Total unique ingredients: {len(ingredient_df)}')
ingredient_df.head(20)


In [ ]:
def simple_plural_to_singular(word):
    # This is a lightweight heuristic for quick exploration, not full lemmatization.
    if word.endswith('ies'):
        return word[:-3] + 'y'
    if word.endswith('es'):
        return word[:-2]
    if word.endswith('s') and len(word) > 3:
        return word[:-1]
    return word

def normalize_ingredients_simple(ingredients):
    # Normalize token-by-token so multi-word ingredients keep their original ordering.
    normalized = []
    for item in ingredients:
        parts = item.split()
        normalized_parts = [simple_plural_to_singular(p) for p in parts]
        normalized.append(' '.join(normalized_parts))
    return normalized

sample_ingredients = all_ingredients[:20]
normalized_ingredients = normalize_ingredients_simple(sample_ingredients)
list(zip(sample_ingredients[:10], normalized_ingredients[:10]))


## 3. Recipe Length Analysis

The notebooks repeatedly examined ingredient counts, direction counts, and rough word counts to identify outliers and decide practical filtering bounds.


In [ ]:
def word_count(text_list):
    if isinstance(text_list, str):
        # Try parsing serialized lists first; otherwise treat the raw string as one item.
        parsed = safe_parse_list(text_list)
        text_list = parsed if parsed else [text_list]
    return sum(len(str(text).split()) for text in text_list)

def list_length_count(list_str):
    return len(safe_parse_list(list_str))

df_profile = df.copy()
df_profile['rough_word_count_directions'] = df_profile['directions'].apply(word_count)
df_profile['rough_word_count_ingredients'] = df_profile['ingredients'].apply(word_count)
df_profile['ingredients_count'] = df_profile['NER'].apply(list_length_count)
df_profile['directions_count'] = df_profile['directions'].apply(list_length_count)

display(df_profile[['rough_word_count_directions', 'rough_word_count_ingredients', 'ingredients_count', 'directions_count']].describe())

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
df_profile['rough_word_count_directions'].hist(bins=50, ax=axes[0, 0])
axes[0, 0].set_title('Directions Word Count')
df_profile['rough_word_count_ingredients'].hist(bins=50, ax=axes[0, 1])
axes[0, 1].set_title('Ingredients Word Count')
df_profile['ingredients_count'].hist(bins=40, ax=axes[1, 0])
axes[1, 0].set_title('Ingredient Count')
df_profile['directions_count'].hist(bins=40, ax=axes[1, 1])
axes[1, 1].set_title('Directions Count')
plt.tight_layout()
plt.show()


In [ ]:
def beautify_print_ingredients(ingredients_str):
    try:
        # Accept both serialized lists from the dataframe and pre-parsed Python lists.
        ingredients_list = safe_parse_list(ingredients_str) if isinstance(ingredients_str, str) else ingredients_str
        print('Items:')
        for i, ingredient in enumerate(ingredients_list, 1):
            print(f'{i}. {ingredient}')
    except Exception as e:
        print('Error formatting ingredients:', e)
        print(ingredients_str)

high_ingredient_examples = df_profile[df_profile['ingredients_count'] > 30]
display(high_ingredient_examples[['title', 'ingredients_count', 'directions_count']].head(10))

if not high_ingredient_examples.empty:
    row_index = high_ingredient_examples.index[0]
    print('Title:', df_profile.loc[row_index, 'title'])
    beautify_print_ingredients(df_profile.loc[row_index, 'ingredients'])


In [ ]:
min_ingredients = 3
max_ingredients = 50
min_steps = 3
max_steps = 30

df_threshold_filtered = df_profile[
    (df_profile['ingredients_count'] > min_ingredients)
    & (df_profile['ingredients_count'] < max_ingredients)
    & (df_profile['directions_count'] > min_steps)
    & (df_profile['directions_count'] < max_steps)
].copy()

print('raw shape:', df_profile.shape)
print('threshold filtered shape:', df_threshold_filtered.shape)
display(df_threshold_filtered[['ingredients_count', 'directions_count']].describe())

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(df_threshold_filtered['directions_count'].values)
ax.set_xlabel('Index')
ax.set_ylabel('directions_count')
ax.set_title('Directions Count After Basic Threshold Filtering')
plt.show()


In [ ]:
filtered_titles = df_threshold_filtered[
    df_threshold_filtered['title'].str.contains(r'\b(directions|test|life)\b', case=False, na=False)
]

print(f"Found {len(filtered_titles)} recipes with suspicious title terms.")
filtered_titles[['title', 'ingredients_count', 'directions_count']].head(20)


## 4. Cleaning and Normalization Pipeline

This section combines the ingredient, title, and direction normalization logic refined across V2 to V4.


In [ ]:
UNIT_VARIATIONS = {
    'cup': ['c', 'c.', 'cup', 'cups'],
    'teaspoon': ['tsp', 'tsp.', 'tsps', 'tsps.', 'teaspoon', 'teaspoons', 't'],
    'tablespoon': ['tbsp', 'tbsp.', 'tbs', 'tbs.', 'tablespoon', 'tablespoons', 'tbl', 'T', 'T.'],
    'ounce': ['oz', 'oz.', 'ounce', 'ounces'],
    'pound': ['lb', 'lb.', 'lbs', 'lbs.', 'pound', 'pounds'],
    'gram': ['g', 'g.', 'gram', 'grams'],
    'kilogram': ['kg', 'kg.', 'kilogram', 'kilograms'],
    'milliliter': ['ml', 'ml.', 'milliliter', 'milliliters'],
    'liter': ['l', 'l.', 'liter', 'liters'],
    'pinch': ['pinch', 'pinches'],
    'dash': ['dash', 'dashes'],
    'package': ['pkg', 'pkg.', 'package', 'packages', 'pack'],
    'slice': ['slice', 'slices'],
    'clove': ['clove', 'cloves'],
    'stick': ['stick', 'sticks'],
    'quart': ['qt', 'qt.', 'quart', 'quarts'],
    'gallon': ['gal', 'gal.', 'gallon', 'gallons'],
}

UNIT_MAP = {}
for canonical, variants in UNIT_VARIATIONS.items():
    for variant in variants:
        UNIT_MAP[variant.lower().rstrip('.')] = canonical

UNIT_PATTERN = re.compile(
    r"\b(" + "|".join(re.escape(k) for k in sorted(set(UNIT_MAP.keys()), key=len, reverse=True)) + r")\b",
    flags=re.IGNORECASE,
)

NOISE_WORDS = [
    r'\(optional\)', r'\boptional\b', r'\bto taste\b', r'\bdivided\b',
    r'\bas needed\b', r'\b(or more)\b', r'\bplus extra\b', r'\bfor garnish\b',
    r'\bpeeled\b', r'\bchilled\b', r'\broom temperature\b', r'\bchopped\b',
    r'\bminced\b', r'\bfreshly\b', r'\bfor serving\b', r'\bor substitute\b',
]
NOISE_RE = re.compile('|'.join(NOISE_WORDS), flags=re.IGNORECASE)

UNICODE_FRACTIONS = {
    '¼': '1/4', '½': '1/2', '¾': '3/4', '⅓': '1/3', '⅔': '2/3',
    '⅕': '1/5', '⅖': '2/5', '⅗': '3/5', '⅘': '4/5', '⅙': '1/6',
    '⅚': '5/6', '⅛': '1/8', '⅜': '3/8', '⅝': '5/8', '⅞': '7/8',
}

def replace_unicode_fractions(s: str) -> str:
    # Convert Unicode fractions early so later regexes only need to handle ASCII forms.
    for uf, ascii_frac in UNICODE_FRACTIONS.items():
        s = s.replace(uf, ascii_frac)
    return s

QUANTITY_AT_START_RE = regex.compile(
    r"^\s*(?P<qty>(?:\d+\s+\d+/\d+)|(?:\d+/\d+)|(?:\d+(?:\.\d+)?)|(?:[" + ''.join(map(re.escape, UNICODE_FRACTIONS.keys())) + r"]))\s*(?P<rest>.*)$",
    flags=regex.IGNORECASE,
)

def normalize_units_in_text(text: str) -> str:
    def _repl(match):
        raw = match.group(1)
        key = raw.lower().rstrip('.')
        return UNIT_MAP.get(key, raw)
    # Collapse unit aliases like `tbsp.` and `T` to one canonical token.
    return UNIT_PATTERN.sub(_repl, text)

def remove_noise(text: str) -> str:
    # Remove optional notes and prep descriptors to keep ingredient names compact.
    text = re.sub(r'\([^)]*\)', '', text)
    text = NOISE_RE.sub('', text)
    return text.strip(' ,')

def extract_and_normalize_quantity(text: str):
    # Separate a leading quantity from the ingredient body when present.
    text = replace_unicode_fractions(text.strip())
    match = QUANTITY_AT_START_RE.match(text)
    if not match:
        return None, text
    return match.group('qty').strip(), match.group('rest').strip()

def normalize_single_ingredient(raw: str, keep_qty: bool = True) -> str:
    if not isinstance(raw, str):
        return raw
    # Standardize punctuation and spacing before extracting quantities and units.
    text = unidecode.unidecode(raw)
    text = text.replace('\u00b0', ' degrees ').replace('\xa0', ' ').strip()
    text = replace_unicode_fractions(text)
    text = re.sub(r'[\u2013\u2014\u2212]', '-', text)
    text = re.sub(r'[^\S\r\n]+', ' ', text)
    text = re.sub(r'^\s*(?:[-\u2022]|(?:\d+[\.\)]))\s+', '', text)

    qty, rest = extract_and_normalize_quantity(text)
    text_rest = remove_noise(rest if qty is not None else text)
    text_rest = normalize_units_in_text(text_rest)
    text_rest = re.sub(r'\s*,\s*', ', ', text_rest).strip(', ').strip()

    # Some strings repeat the unit after the quantity, so peel it off once here.
    first_tokens = text_rest.split()[:2]
    unit_token = None
    for token in first_tokens:
        key = token.lower().rstrip('.')
        if key in UNIT_MAP:
            unit_token = UNIT_MAP[key]
            text_rest = re.sub(rf'^{re.escape(token)}\s*', '', text_rest, flags=re.IGNORECASE)
            break

    parts = []
    if keep_qty and qty is not None:
        parts.append(qty)
    if unit_token:
        parts.append(unit_token)
    parts.append(text_rest)
    normalized = ' '.join(parts).strip()
    return normalized if normalized else text_rest

def normalize_ingredients_list(ingredients: Union[List[str], str], keep_qty: bool = True) -> List[str]:
    if isinstance(ingredients, str):
        stripped = ingredients.strip()
        if stripped.startswith('[') and stripped.endswith(']'):
            try:
                # Prefer parsing real list literals so commas inside ingredient text are preserved.
                parsed = ast.literal_eval(stripped)
                ingredients = parsed if isinstance(parsed, list) else []
            except Exception:
                ingredients = [s.strip() for s in re.split(r'[\n\r,;]+', ingredients) if s.strip()]
        else:
            ingredients = [s.strip() for s in re.split(r'[\n\r,;]+', ingredients) if s.strip()]
    elif isinstance(ingredients, list):
        ingredients = [str(it).strip() for it in ingredients if str(it).strip()]
    else:
        return []

    return [normalize_single_ingredient(raw, keep_qty=keep_qty) for raw in ingredients if normalize_single_ingredient(raw, keep_qty=keep_qty)]

def normalize_title(title: str) -> str:
    # Clean spacing and stray punctuation while keeping the original title wording.
    if not isinstance(title, str):
        return ''
    s = unidecode.unidecode(title).strip()
    s = re.sub(r'\s+', ' ', s)
    s = re.sub(r'^[\(\)]+(?=\w)', '', s).strip()
    s = re.sub(r'[\(\)]+$', '', s).strip()
    if s.count('(') > s.count(')'):
        s += ')' * (s.count('(') - s.count(')'))
    s = re.sub(r'[!?.,;:]+$', '', s)
    return s.title()

def normalize_directions(directions):
    # Join step lists into a single model-friendly string for later token counting.
    if isinstance(directions, list):
        steps = [unidecode.unidecode(str(s)).strip() for s in directions if str(s).strip()]
        return ' '.join(steps)
    s = unidecode.unidecode(str(directions)).strip()
    return re.sub(r'\s+', ' ', s)

def filter_suspicious(df_in: pd.DataFrame) -> pd.DataFrame:
    # Apply coarse bounds to remove rows that are likely malformed or unusable for training.
    out = df_in[df_in['ingredients_normalized'].apply(lambda x: isinstance(x, list) and len(x) >= 2)]
    out = out[out['ingredients_normalized'].apply(lambda x: len(x) <= 100)]
    out = out[out['directions_normalized'].str.len() > 10]
    out = out[out['directions_normalized'].str.len() < 10000]
    out = out[out['input_tokens'] >= 5]
    out = out[out['output_tokens'] >= 10]
    out = out[out['output_tokens'] <= 1500]
    return out

def apply_full_pipeline(df_in: pd.DataFrame) -> pd.DataFrame:
    # Keep the raw dataframe intact and build normalized columns alongside it.
    out = df_in.copy()
    out['title_normalized'] = out['title'].apply(normalize_title)
    out['ingredients_normalized'] = out['ingredients'].apply(lambda x: normalize_ingredients_list(x, keep_qty=True))
    out['directions_normalized'] = out['directions'].apply(normalize_directions)
    out['input_tokens'] = out['ingredients_normalized'].apply(lambda x: len(' '.join(x).split()) if isinstance(x, list) else 0)
    out['output_tokens'] = out['directions_normalized'].apply(lambda x: len(str(x).split()))
    return filter_suspicious(out)


In [ ]:
df_filtered = apply_full_pipeline(df)
df_filtered['ingredients_count'] = df_filtered['NER'].apply(list_length_count)
df_filtered['directions_count'] = df_filtered['directions'].apply(list_length_count)

print('raw shape:', df.shape)
print('cleaned shape:', df_filtered.shape)
display(df_filtered.head())
#df_filtered.describe(include='all').transpose().head(20)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))
ax1.plot(df_filtered['input_tokens'].values)
ax1.set_xlabel('Index')
ax1.set_ylabel('input_tokens')
ax1.set_title('Input Token Counts After Cleaning')

ax2.plot(df_filtered['output_tokens'].values)
ax2.set_xlabel('Index')
ax2.set_ylabel('output_tokens')
ax2.set_title('Output Token Counts After Cleaning')

plt.tight_layout()
plt.show()

#display(df_filtered[['ingredients_count', 'directions_count', 'input_tokens', 'output_tokens']].describe())


In [ ]:
sample_cols = [
    'title', 'title_normalized', 'ingredients', 'ingredients_normalized',
    'directions', 'directions_normalized', 'ingredients_count', 'directions_count'
]
df_filtered[sample_cols].head(5)
